In [2]:
import pandas as pd
pd.set_option("display.float_format", "{:,.2f}".format)

df = pd.read_csv("../data/paysim/PS_20174392719_1491204439457_log.csv")
print(df.shape)
df.head()

(6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,"9,839.64",C1231006815,"170,136.00","160,296.36",M1979787155,0.00,0.00,0,0
1,1,PAYMENT,"1,864.28",C1666544295,"21,249.00","19,384.72",M2044282225,0.00,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,"21,182.00",0.00,1,0
4,1,PAYMENT,"11,668.14",C2048537720,"41,554.00","29,885.86",M1230701703,0.00,0.00,0,0


In [5]:
print("Fraud rate:", df["isFraud"].mean())
print(df["isFraud"].value_counts())
df.groupby("type")["isFraud"].agg(rows="count", frauds="sum")

Fraud rate: 0.001290820448180152
isFraud
0    6354407
1       8213
Name: count, dtype: int64


,rows,frauds
type,,
CASH_IN,1399284,0
CASH_OUT,2237500,4116
DEBIT,41432,0
PAYMENT,2151495,0
TRANSFER,532909,4097


In [4]:
print(df.isna().sum())
df.dtypes

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
errorOrig         0
errorDest         0
dtype: int64


step                int64
type                  str
amount            float64
nameOrig              str
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest              str
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
errorOrig         float64
errorDest         float64
dtype: object

In [3]:
df["errorOrig"] = df["newbalanceOrig"] + df["amount"] - df["oldbalanceOrg"]
df["errorDest"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]
df.groupby("isFraud")[["amount", "oldbalanceOrg", "newbalanceOrig", "errorOrig", "errorDest"]].mean()

,amount,oldbalanceOrg,newbalanceOrig,errorOrig,errorDest
isFraud,,,,,
0,"178,197.04","832,828.71","855,970.23","201,338.56","54,692.23"
1,"1,467,967.30","1,649,667.61","192,392.63","10,692.33","732,509.30"


In [6]:
t = df[df["type"].isin(["TRANSFER", "CASH_OUT"])]
t["emptied"] = (t["newbalanceOrig"] == 0) & (t["oldbalanceOrg"] > 0)
t.groupby("emptied")["isFraud"].agg(rows="count", frauds="sum", rate="mean")

,rows,frauds,rate
emptied,,,
False,1582335,201,0.00
True,1188074,8012,0.01


In [7]:
t["destZero"] = (t["newbalanceDest"] == 0) & (t["amount"] > 0)
t.groupby("destZero")["isFraud"].agg(rows="count", frauds="sum", rate="mean")

,rows,frauds,rate
destZero,,,
False,2753816,4128,0.00
True,16593,4085,0.25


In [8]:
df.groupby("step")["isFraud"].sum().describe()
pd.crosstab(df["isFraud"], df["isFlaggedFraud"])
print(df["nameOrig"].nunique(), df["nameDest"].nunique())

6353307 2722362


In [17]:
d = df[df["type"].isin(["TRANSFER", "CASH_OUT"])].copy()

d["errorOrig"] = d["oldbalanceOrg"] - d["amount"] - d["newbalanceOrig"]
d["errorDest"] = d["oldbalanceDest"] + d["amount"] - d["newbalanceDest"]
d["emptied"] = ((d["newbalanceOrig"] == 0) & (d["oldbalanceOrg"] > 0)).astype(int)
d["destZero"] = ((d["newbalanceDest"] == 0) & (d["amount"] > 0)).astype(int)
d["isTransfer"] = (d["type"] == "TRANSFER").astype(int)
d["hour"] = d["step"] % 24

feats = ["amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest",
         "errorOrig", "errorDest", "emptied", "destZero", "isTransfer", "hour"]

In [10]:
train = d[d["step"] <= 520]
val   = d[(d["step"] > 520) & (d["step"] <= 630)]
test  = d[d["step"] > 630]
for name, s in [("train", train), ("val", val), ("test", test)]:
    print(name, len(s), s["isFraud"].sum())

train 2653729 5781
val 78689 1168
test 37991 1264


In [18]:
import numpy as np
from sklearn.metrics import average_precision_score, roc_curve, confusion_matrix

def evaluate(y, p, name, thr=0.5):
    fpr, tpr, _ = roc_curve(y, p)
    rec_at_1 = tpr[fpr <= 0.01].max()
    print(f"{name}: PR-AUC={average_precision_score(y, p):.4f} | recall@1%FPR={rec_at_1:.4f}")
    print(confusion_matrix(y, p >= thr))

In [14]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lr = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000))
lr.fit(train[feats], train["isFraud"])
evaluate(test["isFraud"], lr.predict_proba(test[feats])[:, 1], "LogReg")

LogReg: PR-AUC=0.9395 | recall@1%FPR=0.9106
[[35621  1106]
 [   51  1213]]


In [15]:
import xgboost as xgb

pos_weight = (train["isFraud"] == 0).sum() / (train["isFraud"] == 1).sum()
xgb_model = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=pos_weight, tree_method="hist",
    eval_metric="aucpr", early_stopping_rounds=30, n_jobs=-1,
)
xgb_model.fit(train[feats], train["isFraud"],
              eval_set=[(val[feats], val["isFraud"])], verbose=50)
evaluate(test["isFraud"], xgb_model.predict_proba(test[feats])[:, 1], "XGBoost")

[0]	validation_0-aucpr:0.99937
[39]	validation_0-aucpr:0.99996
XGBoost: PR-AUC=1.0000 | recall@1%FPR=1.0000
[[36726     1]
 [    1  1263]]


In [16]:
pd.Series(xgb_model.feature_importances_, index=feats).sort_values(ascending=False)

errorOrig        0.61
emptied          0.13
newbalanceOrig   0.11
oldbalanceOrg    0.06
destZero         0.03
newbalanceDest   0.02
amount           0.01
isTransfer       0.01
errorDest        0.01
hour             0.01
oldbalanceDest   0.00
dtype: float32

In [19]:
d["drainRatio"] = d["amount"] / (d["oldbalanceOrg"] + 1)
d["destOldZero"] = (d["oldbalanceDest"] == 0).astype(int)
feats_rt = ["amount", "oldbalanceOrg", "oldbalanceDest", "drainRatio", "destOldZero", "isTransfer", "hour"]

# re-split, because d gained new columns
train = d[d["step"] <= 520]
val   = d[(d["step"] > 520) & (d["step"] <= 630)]
test  = d[d["step"] > 630]

xgb_rt = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight,
    tree_method="hist", eval_metric="aucpr", early_stopping_rounds=30, n_jobs=-1,
)
xgb_rt.fit(train[feats_rt], train["isFraud"],
           eval_set=[(val[feats_rt], val["isFraud"])], verbose=0)
evaluate(test["isFraud"], xgb_rt.predict_proba(test[feats_rt])[:, 1], "XGBoost (pre-transaction only)")
pd.Series(xgb_rt.feature_importances_, index=feats_rt).sort_values(ascending=False)

XGBoost (pre-transaction only): PR-AUC=0.9962 | recall@1%FPR=1.0000
[[36656    71]
 [    1  1263]]


drainRatio       0.73
oldbalanceOrg    0.12
isTransfer       0.04
amount           0.04
destOldZero      0.03
oldbalanceDest   0.02
hour             0.01
dtype: float32

In [20]:
p = xgb_rt.predict_proba(test[feats_rt])[:, 1]
y = test["isFraud"].values
COST_MISSED, COST_FALSE_ALARM = 100, 5   # assumptions, state them in the README

rows = []
for thr in np.linspace(0.05, 0.95, 19):
    pred = p >= thr
    fn = ((~pred) & (y == 1)).sum()
    fp = (pred & (y == 0)).sum()
    rows.append((round(thr, 2), fn, fp, fn * COST_MISSED + fp * COST_FALSE_ALARM))
cost_df = pd.DataFrame(rows, columns=["threshold", "missed", "false_alarms", "total_cost"])
cost_df.sort_values("total_cost").head(5)

,threshold,missed,false_alarms,total_cost
16,0.85,1,69,445
15,0.80,1,70,450
14,0.75,1,70,450
13,0.70,1,70,450
8,0.45,1,71,455


In [24]:
import shap
explainer = shap.TreeExplainer(xgb_rt)
sample = test[feats_rt].sample(2000, random_state=0)
sv = explainer(sample)
shap.plots.beeswarm(sv)                      # global view

fraud_row = test[test["isFraud"] == 1][feats_rt].iloc[[0]]
shap.plots.waterfall(explainer(fraud_row)[0])   # why this one was flagged

ImportError: matplotlib is not installed so plotting is not available! Run `pip install matplotlib` to fix this.

In [23]:
import json
xgb_rt.save_model("../models/mobile_money_xgb.json")
json.dump({"features": feats_rt, "threshold": 0.5}, open("../models/mobile_money_meta.json", "w"))